# 面试题：链接预测怎样做负采样，如何避免假负例和时间泄漏？

## 可以直接复述的回答

隐式反馈图里“没有边”不等于负反馈，它可能只是用户尚未看到，因此从全量未连接边均匀抽样会制造假负例。时间切分后，未来新增的正边尤其容易被错误当作训练负例；但用未来正边名单把它们排除又会把测试信息泄漏到训练。更稳妥的负样本来自截止时刻前已经曝光但未点击的边，并且要记录曝光策略带来的选择偏差。链接预测模型可以用用户和商品 embedding 点积打分，以正边分数高于曝光负边的 BPR 目标训练。评估时应屏蔽训练已购商品，在同一候选全集上看 held-out 边的排名，而不是只对采样负例做二分类。本题用 6 个用户、10 个商品、18 条历史正边和 6 条未来购买，实际训练 embedding 并展示假负例、泄漏修复误区和正确采样。

## 真实案例

用户和商品按品类形成摄影、运动、咖啡三类偏好；每个用户有截止日前购买、曝光未点和截止日后购买。全部是脱敏教学 ID，不能代表真实推荐分布。

In [1]:
from pprint import pprint  # 导入结构化打印函数以展示边和排名
import torch  # 导入 PyTorch 以训练真实链接预测参数
torch.manual_seed(23)  # 固定随机初始化保证训练结果可复现
torch.set_num_threads(1)  # 限制线程数以稳定小型实验
users = ["U1", "U2", "U3", "U4", "U5", "U6"]  # 定义六个脱敏用户节点
items = ["I1相机", "I2镜头", "I3跑鞋", "I4运动袜", "I5咖啡机", "I6磨豆机", "I7会员", "I8三脚架", "I9骑行包", "I10手冲壶"]  # 定义十个带业务语义的商品节点
train_positives = {"U1": ["I1相机", "I7会员", "I8三脚架"], "U2": ["I1相机", "I2镜头", "I7会员"], "U3": ["I3跑鞋", "I7会员", "I9骑行包"], "U4": ["I3跑鞋", "I4运动袜", "I7会员"], "U5": ["I5咖啡机", "I7会员", "I10手冲壶"], "U6": ["I5咖啡机", "I6磨豆机", "I7会员"]}  # 构造截止日前十八条购买正边
future_positives = {"U1": "I2镜头", "U2": "I8三脚架", "U3": "I4运动袜", "U4": "I9骑行包", "U5": "I6磨豆机", "U6": "I10手冲壶"}  # 构造截止日后的六条评估正边
exposed_not_clicked = {"U1": ["I3跑鞋", "I5咖啡机"], "U2": ["I4运动袜", "I6磨豆机"], "U3": ["I1相机", "I5咖啡机"], "U4": ["I2镜头", "I6磨豆机"], "U5": ["I1相机", "I3跑鞋"], "U6": ["I2镜头", "I4运动袜"]}  # 构造截止日前真实曝光未点边
print("历史正边与未来评估边：")  # 输出真实交互标题
pprint([{"用户": user, "历史购买": train_positives[user], "未来购买": future_positives[user], "曝光未点": exposed_not_clicked[user]} for user in users])  # 展示六个用户的时间切分记录
print("历史正边数量：", sum(len(values) for values in train_positives.values()))  # 输出训练图规模

历史正边与未来评估边：
[{'历史购买': ['I1相机', 'I7会员', 'I8三脚架'],
  '曝光未点': ['I3跑鞋', 'I5咖啡机'],
  '未来购买': 'I2镜头',
  '用户': 'U1'},
 {'历史购买': ['I1相机', 'I2镜头', 'I7会员'],
  '曝光未点': ['I4运动袜', 'I6磨豆机'],
  '未来购买': 'I8三脚架',
  '用户': 'U2'},
 {'历史购买': ['I3跑鞋', 'I7会员', 'I9骑行包'],
  '曝光未点': ['I1相机', 'I5咖啡机'],
  '未来购买': 'I4运动袜',
  '用户': 'U3'},
 {'历史购买': ['I3跑鞋', 'I4运动袜', 'I7会员'],
  '曝光未点': ['I2镜头', 'I6磨豆机'],
  '未来购买': 'I9骑行包',
  '用户': 'U4'},
 {'历史购买': ['I5咖啡机', 'I7会员', 'I10手冲壶'],
  '曝光未点': ['I1相机', 'I3跑鞋'],
  '未来购买': 'I6磨豆机',
  '用户': 'U5'},
 {'历史购买': ['I5咖啡机', 'I6磨豆机', 'I7会员'],
  '曝光未点': ['I2镜头', 'I4运动袜'],
  '未来购买': 'I10手冲壶',
  '用户': 'U6'}]
历史正边数量： 18


## Baseline / 基线：按历史商品流行度排序

流行度不使用用户信息。评估时先屏蔽该用户已经购买的商品，再在其余商品中查找未来购买的真实排名。

In [2]:
popularity = {item: sum(item in train_positives[user] for user in users) for item in items}  # 统计每个商品的历史购买用户数
def popularity_ranking(user):  # 定义屏蔽已购商品的流行度基线
    candidates = [item for item in items if item not in train_positives[user]]  # 构造用户未购买候选全集
    return sorted(candidates, key=lambda item: (-popularity[item], item))  # 按流行度降序和商品名稳定排序
baseline_rankings = {user: popularity_ranking(user) for user in users}  # 为六个用户运行同一基线
baseline_rows = []  # 创建逐用户基线排名表
for user in users:  # 遍历全部评估用户
    rank = baseline_rankings[user].index(future_positives[user]) + 1  # 查找未来真实购买的排名
    baseline_rows.append({"用户": user, "未来购买": future_positives[user], "Top3": baseline_rankings[user][:3], "真实排名": rank})  # 保存候选与排名
baseline_mrr = sum(1 / row["真实排名"] for row in baseline_rows) / len(baseline_rows)  # 计算流行度基线 MRR
print("Popularity Baseline：")  # 输出基线结果标题
pprint(baseline_rows)  # 展示每个用户的未来边排名
print(f"Baseline MRR={baseline_mrr:.4f}")  # 输出基线聚合指标

Popularity Baseline：
[{'Top3': ['I3跑鞋', 'I5咖啡机', 'I10手冲壶'], '未来购买': 'I2镜头', '用户': 'U1', '真实排名': 4},
 {'Top3': ['I3跑鞋', 'I5咖啡机', 'I10手冲壶'], '未来购买': 'I8三脚架', '用户': 'U2', '真实排名': 6},
 {'Top3': ['I1相机', 'I5咖啡机', 'I10手冲壶'], '未来购买': 'I4运动袜', '用户': 'U3', '真实排名': 5},
 {'Top3': ['I1相机', 'I5咖啡机', 'I10手冲壶'], '未来购买': 'I9骑行包', '用户': 'U4', '真实排名': 7},
 {'Top3': ['I1相机', 'I3跑鞋', 'I2镜头'], '未来购买': 'I6磨豆机', '用户': 'U5', '真实排名': 5},
 {'Top3': ['I1相机', 'I3跑鞋', 'I10手冲壶'], '未来购买': 'I10手冲壶', '用户': 'U6', '真实排名': 3}]
Baseline MRR=0.2155


## 失败案例：未连接边、假负例与未来信息泄漏

朴素采样器按商品列表取第一个未连接边；U1 的第一个正是未来会购买的镜头，所以它被错误标成负例。若用 future_positives 排除它，虽然避免了假负例，却已经读取测试答案。正确训练负例只来自截止日前的曝光未点日志。

In [3]:
def naive_unconnected_negative(user):  # 定义把未知边当负例的错误采样器
    return next(item for item in items if item not in train_positives[user])  # 返回第一个未连接但未必负反馈的商品
naive_negatives = {user: naive_unconnected_negative(user) for user in users}  # 对六个用户执行错误负采样
false_negative_users = [user for user in users if naive_negatives[user] == future_positives[user]]  # 找出被错误采成负例的未来正边
future_aware_negatives = {user: next(item for item in items if item not in train_positives[user] and item != future_positives[user]) for user in users}  # 错误地利用未来标签排除假负例
safe_negative_pairs = [(user, item) for user in users for item in exposed_not_clicked[user]]  # 只用截止日前曝光未点构造可审计负边
print("朴素未连接负样本：", naive_negatives)  # 展示未知边采样结果
print("其中实际是未来正边的用户：", false_negative_users)  # 展示假负例证据
print("看似修复但发生泄漏的负样本：", future_aware_negatives)  # 展示读取未来标签的错误修复
print("正确的截止日前曝光负边：")  # 输出安全采样标题
pprint(safe_negative_pairs)  # 展示十二条有曝光证据的训练负边

朴素未连接负样本： {'U1': 'I2镜头', 'U2': 'I3跑鞋', 'U3': 'I1相机', 'U4': 'I1相机', 'U5': 'I1相机', 'U6': 'I1相机'}
其中实际是未来正边的用户： ['U1']
看似修复但发生泄漏的负样本： {'U1': 'I3跑鞋', 'U2': 'I3跑鞋', 'U3': 'I1相机', 'U4': 'I1相机', 'U5': 'I1相机', 'U6': 'I1相机'}
正确的截止日前曝光负边：
[('U1', 'I3跑鞋'),
 ('U1', 'I5咖啡机'),
 ('U2', 'I4运动袜'),
 ('U2', 'I6磨豆机'),
 ('U3', 'I1相机'),
 ('U3', 'I5咖啡机'),
 ('U4', 'I2镜头'),
 ('U4', 'I6磨豆机'),
 ('U5', 'I1相机'),
 ('U5', 'I3跑鞋'),
 ('U6', 'I2镜头'),
 ('U6', 'I4运动袜')]


## 手写 embedding 链接预测与真实 BPR 反向传播

模型参数就是用户矩阵与商品矩阵，forward 取对应行后做点积。每个历史正边分别与该用户的两条曝光负边比较，BPR loss 用 logaddexp 稳定计算 softplus(-margin)。

In [4]:
user_to_index = {user: index for index, user in enumerate(users)}  # 建立用户 ID 到参数行号的映射
item_to_index = {item: index for index, item in enumerate(items)}  # 建立商品 ID 到参数行号的映射
class DotLinkPredictor(torch.nn.Module):  # 定义最小双部图点积模型
    def __init__(self, user_count, item_count, dimension):  # 初始化用户和商品 embedding
        super().__init__()  # 初始化 PyTorch 模块基类
        self.user_vectors = torch.nn.Parameter(torch.randn(user_count, dimension) * 0.1)  # 创建可训练用户向量矩阵
        self.item_vectors = torch.nn.Parameter(torch.randn(item_count, dimension) * 0.1)  # 创建可训练商品向量矩阵
    def forward(self, user_indices, item_indices):  # 定义边存在性的点积分数
        user_batch = self.user_vectors[user_indices]  # 取出批次用户向量
        item_batch = self.item_vectors[item_indices]  # 取出批次商品向量
        return (user_batch * item_batch).sum(dim=1)  # 逐维相乘求和得到链接分数
training_triplets = []  # 创建正边与曝光负边的成对训练样本
for user in users:  # 遍历六个用户
    for positive in train_positives[user]:  # 遍历该用户截止日前正边
        for negative in exposed_not_clicked[user]:  # 遍历同用户真实曝光未点负边
            training_triplets.append((user_to_index[user], item_to_index[positive], item_to_index[negative]))  # 保存用户、正商品、负商品索引
triplet_tensor = torch.tensor(training_triplets, dtype=torch.long)  # 把三元组列表转换为训练张量
model = DotLinkPredictor(len(users), len(items), 6)  # 实例化六维点积链接预测器
training_ledger = []  # 创建 loss、margin 与梯度账本
for epoch in range(501):  # 执行五百零一次全批量训练
    positive_scores = model(triplet_tensor[:, 0], triplet_tensor[:, 1])  # 对历史正边运行真实 forward
    negative_scores = model(triplet_tensor[:, 0], triplet_tensor[:, 2])  # 对曝光负边运行真实 forward
    margins = positive_scores - negative_scores  # 计算正边相对负边的分数优势
    loss = torch.logaddexp(torch.zeros_like(margins), -margins).mean()  # 手写数值稳定的 BPR softplus 损失
    loss.backward()  # 运行真实 backward 计算 embedding 梯度
    gradient_norm = float(model.user_vectors.grad.norm())  # 读取用户向量梯度范数
    if epoch % 100 == 0:  # 每一百轮记录训练状态
        training_ledger.append({"epoch": epoch, "loss": round(float(loss), 5), "平均margin": round(float(margins.mean()), 5), "用户梯度": round(gradient_norm, 5)})  # 保存关键训练中间量
    with torch.no_grad():  # 关闭手动更新阶段的梯度记录
        model.user_vectors -= 0.08 * model.user_vectors.grad  # 手动更新用户 embedding
        model.item_vectors -= 0.08 * model.item_vectors.grad  # 手动更新商品 embedding
    model.user_vectors.grad.zero_()  # 清空用户参数梯度
    model.item_vectors.grad.zero_()  # 清空商品参数梯度
print("BPR loss、margin 与真实梯度：")  # 输出训练过程标题
pprint(training_ledger)  # 展示正负边间隔随训练扩大

BPR loss、margin 与真实梯度：
[{'epoch': 0, 'loss': 0.68788, '平均margin': 0.01092, '用户梯度': 0.05151},
 {'epoch': 100, 'loss': 0.64637, '平均margin': 0.09784, '用户梯度': 0.06293},
 {'epoch': 200, 'loss': 0.56428, '平均margin': 0.28947, '用户梯度': 0.08563},
 {'epoch': 300, 'loss': 0.42706, '平均margin': 0.67811, '用户梯度': 0.09961},
 {'epoch': 400, 'loss': 0.27533, '平均margin': 1.25226, '用户梯度': 0.09298},
 {'epoch': 500, 'loss': 0.16398, '平均margin': 1.87253, '用户梯度': 0.07321}]


## 同一候选全集评估与结果解读

评估不只和训练时采到的两个负例比较，而是在全部未购买商品中排序。成对用户通过共同历史商品形成相近偏好，所以模型可以把另一位同类用户买过的 held-out 商品排到前面。

In [5]:
def model_ranking(user):  # 定义全候选商品链接排名函数
    candidate_items = [item for item in items if item not in train_positives[user]]  # 屏蔽该用户训练期已购买商品
    user_indices = torch.tensor([user_to_index[user]] * len(candidate_items), dtype=torch.long)  # 构造重复用户索引批次
    item_indices = torch.tensor([item_to_index[item] for item in candidate_items], dtype=torch.long)  # 构造全部候选商品索引
    with torch.no_grad():  # 关闭评估阶段梯度记录
        scores = model(user_indices, item_indices)  # 实算每条候选边的点积分数
    rows = [{"item": item, "score": round(float(scores[index]), 4)} for index, item in enumerate(candidate_items)]  # 保存商品和链接分数
    return sorted(rows, key=lambda row: (-row["score"], row["item"]))  # 返回稳定的全候选排名
model_rankings = {user: model_ranking(user) for user in users}  # 为六个用户执行链接预测
result_rows = []  # 创建逐用户结果表
for user in users:  # 遍历全部评估用户
    ranked_items = [row["item"] for row in model_rankings[user]]  # 提取商品编号排名
    rank = ranked_items.index(future_positives[user]) + 1  # 查找未来真实边的排名
    baseline_rank = next(row["真实排名"] for row in baseline_rows if row["用户"] == user)  # 读取同用户流行度排名
    result_rows.append({"用户": user, "未来购买": future_positives[user], "Popularity排名": baseline_rank, "LinkPrediction排名": rank, "模型Top3": model_rankings[user][:3]})  # 保存逐用户同口径比较
model_mrr = sum(1 / row["LinkPrediction排名"] for row in result_rows) / len(result_rows)  # 计算链接预测 MRR
print("逐用户未来边排名：")  # 输出结果表标题
pprint(result_rows)  # 展示每个 held-out 正边的完整排名变化
print(f"MRR 从 {baseline_mrr:.4f} 变为 {model_mrr:.4f}")  # 输出同一候选全集的指标对照

逐用户未来边排名：
[{'LinkPrediction排名': 2,
  'Popularity排名': 4,
  '未来购买': 'I2镜头',
  '模型Top3': [{'item': 'I10手冲壶', 'score': 0.3476},
             {'item': 'I2镜头', 'score': 0.29},
             {'item': 'I9骑行包', 'score': 0.2569}],
  '用户': 'U1'},
 {'LinkPrediction排名': 3,
  'Popularity排名': 6,
  '未来购买': 'I8三脚架',
  '模型Top3': [{'item': 'I9骑行包', 'score': 0.2156},
             {'item': 'I3跑鞋', 'score': 0.0502},
             {'item': 'I8三脚架', 'score': -0.0717}],
  '用户': 'U2'},
 {'LinkPrediction排名': 3,
  'Popularity排名': 5,
  '未来购买': 'I4运动袜',
  '模型Top3': [{'item': 'I2镜头', 'score': 0.6084},
             {'item': 'I8三脚架', 'score': 0.4358},
             {'item': 'I4运动袜', 'score': 0.1584}],
  '用户': 'U3'},
 {'LinkPrediction排名': 3,
  'Popularity排名': 7,
  '未来购买': 'I9骑行包',
  '模型Top3': [{'item': 'I10手冲壶', 'score': 0.249},
             {'item': 'I8三脚架', 'score': 0.0561},
             {'item': 'I9骑行包', 'score': 0.014}],
  '用户': 'U4'},
 {'LinkPrediction排名': 3,
  'Popularity排名': 5,
  '未来购买': 'I6磨豆机',
  '模型Top3': [{'ite

## 失败修正总结

假负例的正确修复不是偷看未来，而是改善数据采集：记录曝光、位置、时间和未点击。若没有曝光日志，应把样本称为“未观察边”，采用 PU learning、置信权重或更保守的采样，并在评估报告中说明假负例风险。

## 生产差距

线上还需按时间构图、处理新用户新商品、按流行度或度数校正负采样、进行 hard-negative mining，并防止同一会话或未来交互泄漏。训练和在线候选 ID 必须版本一致，评估需包含 Recall@K、MRR、覆盖率和分群结果，最终仍要由在线实验验证。

In [6]:
assert len(false_negative_users) >= 1  # 验证朴素未连接采样确实制造未来假负例
assert len(safe_negative_pairs) == 12  # 验证正确负边均来自十二条截止日前曝光未点记录
assert training_ledger[-1]["loss"] < training_ledger[0]["loss"]  # 验证真实 BPR 反向传播降低训练损失
assert training_ledger[-1]["平均margin"] > training_ledger[0]["平均margin"]  # 验证正边相对负边优势扩大
assert model_mrr > baseline_mrr  # 验证链接预测优于同候选集流行度基线
assert all(row["LinkPrediction排名"] <= 3 for row in result_rows)  # 验证六条未来正边都进入模型前三名
print("最小回归测试通过：时间切分、曝光负采样、真实训练与全候选评估均满足预期")  # 输出集中断言的验收结论

最小回归测试通过：时间切分、曝光负采样、真实训练与全候选评估均满足预期
